In [2]:
import pandas
import requests
import octoparse 
import json
import time


In [10]:
Task_ID = pandas.read_csv('/Users/jaapjanlammers/Desktop/Freelancedirectory/Important_allGigs/automation_details.csv', sep= ';')
base_url = 'https://openapi.octoparse.com/'
username = 'jj@nineways.nl'
password = 'rLyQiH2Th&8Ct4IX'

In [12]:
access_token = 'eyJhbGciOiJSUzI1NiIsImtpZCI6ImdXS3hWc0F4SHhaMDlDcWh6NkNJb3ciLCJ0eXAiOiJhdCtqd3QifQ.eyJuYmYiOjE3NDY1NTc4OTEsImV4cCI6MTc0NjY0NDI5MSwiaXNzIjoiaHR0cHM6Ly9pZGVudGl0eS5vY3RvcGFyc2UuY29tIiwiY2xpZW50X2lkIjoiT3BlbkFwaSIsInN1YiI6IjUwMjQ0ZTQ4LTVlNTUtNGRiMC05ZjdiLTZhODQ2OTRhYzA0MyIsImF1dGhfdGltZSI6MTc0NjU1Nzg5MSwiaWRwIjoibG9jYWwiLCJodHRwOi8vc2NoZW1hcy54bWxzb2FwLm9yZy93cy8yMDA1LzA1L2lkZW50aXR5L2NsYWltcy9uYW1laWRlbnRpZmllciI6IjUwMjQ0ZTQ4LTVlNTUtNGRiMC05ZjdiLTZhODQ2OTRhYzA0MyIsImh0dHA6Ly9zY2hlbWFzLnhtbHNvYXAub3JnL3dzLzIwMDUvMDUvaWRlbnRpdHkvY2xhaW1zL25hbWUiOiJMYW1tZXJzIiwicHJlZmVycmVkX3VzZXJuYW1lIjoiTGFtbWVycyIsInVuaXF1ZV9uYW1lIjoiTGFtbWVycyIsInJlZ2lzdGVyX2RhdGUiOiIyMDI0LTA5LTE2VDIwOjAxOjI4KzAwOjAwIiwibGFzdF9sb2dpbl9kYXRlIjoiMjAyNS0wNS0wNlQxMDozNjozOCswMDowMCIsImxhc3RfcGFzc3dvcmRfY2hhbmdlZF9kYXRlIjoiMjAyNS0wMS0yMlQxMTo1NjoxMCswMDowMCIsImh0dHA6Ly9zY2hlbWFzLnhtbHNvYXAub3JnL3dzLzIwMDUvMDUvaWRlbnRpdHkvY2xhaW1zL2VtYWlsYWRkcmVzcyI6ImpqQG5pbmV3YXlzLm5sIiwiZW1haWwiOiJqakBuaW5ld2F5cy5ubCIsImVtYWlsX3ZlcmlmaWVkIjp0cnVlLCJzY29wZSI6WyJvcGVuaWQiLCJwcm9maWxlIiwib2ZmbGluZV9hY2Nlc3MiXSwiYW1yIjpbInB3ZCJdfQ.GXFN_pc8RK1lU-7aVTh6YPjCfBvGSdtQbxFqhz-pS5X2kqBFAIFRLlljY9UFHqs90wst43mF1hsp2JAkJHCWcR8ag1ifa2zrC6MaaFph-QsHyDgPjj0X_ovmXqGPzdVbmmV8tbKp-G1cwXlw6wxxqH_8xHJpGhhgLO-fgXOHDIaxRyOouJcavTEXsOO79bjgPkkPKOIrF1h8n4Mha8GbGQO20NrmJ3H5Pea0Zo2hauPHRzse5X57agtu1NronQpDpmji52oOXMZxBWLaAo1KAsZF6oftPaKSwI8xu8m9F9KXNFWjyqnB1E5oi65HaLR5gLQ_4uaPWBNgQl-1TOXPVg'
refresh_token = '1Iiu_iNd1yOvVh9wgBGrOoytBWnP4aCbfOwdtMFENZk'

In [ ]:
def log_in(base_url, username, password):
    print('Get token:')
    token_url = base_url + 'token'  # Ensure this is the correct endpoint
    headers = {'Content-Type': 'application/json'}
    payload = json.dumps({
        'username': username,
        'password': password,
        'grant_type': 'password'
    })

    try:
        response = requests.post(token_url, headers=headers, data=payload)
        response.raise_for_status()  # Raise an exception for bad status codes
        token_entity = response.json()
        print("JSON Response:")
        print(token_entity)
        if 'data' in token_entity and 'access_token' in token_entity['data'] and 'refresh_token' in token_entity['data']:
            global access_token
            global refresh_token
            access_token = token_entity['data']['access_token']
            refresh_token = token_entity['data']['refresh_token']
            print(f"Successfully retrieved access token: {access_token[:20]}...") # Print a snippet
            print(f"Successfully retrieved refresh token: {refresh_token[:20]}...") # Print a snippet
            return True
        else:
            print("Access and/or refresh token not found in the 'data' section of the response.")
            return False
    except requests.exceptions.RequestException as e:
        print(f"An error occurred during the token request: {e}")
        print(f"HTTP Status Code: {response.status_code if 'response' in locals() else 'N/A'}")
        print("Raw Response Content:")
        print(response.text if 'response' in locals() else 'N/A')
        return False
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e}")
        print("Raw Response Content:")
        print(response.text if 'response' in locals() else 'N/A')
        return False
        
def refresh_token_function(base_url, refresh_token_param): # Use a different parameter name
    try:
        headers = {
            'Content-Type': 'application/x-www-form-urlencoded'
        }
        refresh_url = f"{base_url}token"
        print(f"Refreshing token:")

        data = {
            'refresh_token': refresh_token_param, # Use the parameter name here
            'grant_type': 'refresh_token'
        }

        response = requests.post(refresh_url, headers=headers, data=data)
        print(f"HTTP Status Code: {response.status_code}")
        print(f"Raw Response Content:\n{response.text}")

        if response.status_code == 200:
            global access_token_global, refresh_token_global # Refer to the renamed globals
            data = response.json()
            access_token_global = data.get('access_token')
            refresh_token_global = data.get('refresh_token')
            print(f"Token refreshed successfully. New access token: {access_token_global[:30]}...")
            if refresh_token_global:
                print(f"New refresh token: {refresh_token_global[:30]}...")
            return True
        else:
            print(f"Token refresh failed with status code: {response.status_code}")
            return False
    except Exception as e:
        print(f"An error occurred during the token refresh request: {str(e)}")
        return False

In [ ]:
log_in(base_url, username, password)

In [ ]:
def verify_task_exists(base_url, access_token, task_id):
    """
    Verify if a task exists before attempting to clear it.
    
    Args:
        base_url (str): Base URL of the API
        access_token (str): Current access token
        task_id (str): ID of the task to verify
        
    Returns:
        bool: True if task exists, False otherwise
    """
    try:
        headers = {
            'Authorization': f'Bearer {access_token}'
        }
        
        # Ensure base_url ends with a slash
        if not base_url.endswith('/'):
            base_url += '/'
            
        api_url = f"{base_url}api/v1/tasks/{task_id}"
        print(f"Verifying task ID: {task_id}")
        
        response = requests.get(api_url, headers=headers)
        
        if response.status_code == 200:
            print(f"Task ID {task_id} exists and is valid")
            return True
        elif response.status_code == 404:
            print(f"Task ID {task_id} does not exist or has been deleted")
            return False
        else:
            print(f"Error verifying task: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except Exception as e:
        print(f"Error verifying task: {str(e)}")
        return False

def clear_task_data(base_url, access_token, task_id, task_name):
    """
    Clear a specific task using the API.
    
    Args:
        base_url (str): Base URL of the API
        access_token (str): Current access token
        task_id (str): ID of the task to clear
        task_name (str): Name of the task (for logging)
        
    Returns:
        bool: True if task was cleared successfully
    """
    try:
        # First verify the task exists
        if not verify_task_exists(base_url, access_token, task_id):
            print(f"Skipping task '{task_name}' as it does not exist")
            return False
            
        headers = {
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {access_token}'
        }
        
        # Ensure base_url ends with a slash
        if not base_url.endswith('/'):
            base_url += '/'
            
        api_url = f"{base_url}api/v1/tasks/{task_id}/clear"
        print(f"Attempting to clear task with ID: {task_id}")
        print(f"Using URL: {api_url}")
        
        response = requests.post(
            api_url,
            headers=headers
        )
        
        if response.status_code == 200:
            print(f"Successfully cleared task: {task_name}")
            return True
        elif response.status_code == 404:
            print(f"Task not found. Please verify the task ID: {task_id}")
            print(f"Response content: {response.text}")
            return False
        else:
            print(f"Failed to clear task '{task_name}'. Status code: {response.status_code}")
            print(f"Response content: {response.text}")
            return False
            
    except Exception as e:
        print(f"Error in clear_task_data: {str(e)}")
        return False

def Clear_task(Task_ID):
    """
    Clear multiple tasks with automatic token refresh and retry logic.
    
    Args:
        Task_ID (dict): Dictionary containing Task_ID and Task_name
    """
    global access_token, refresh_token
    
    print("\nTask_ID dictionary contents:")
    print(f"Keys: {Task_ID.keys()}")
    print(f"Task_ID values: {Task_ID['Task_ID']}")
    print(f"Task_name values: {Task_ID['Task_name']}")
    
    for index, Task in enumerate(Task_ID['Task_ID']):
        Task_ID_name = Task_ID['Task_name'].iloc[index]
        print(f"\nProcessing task: {Task_ID_name}")
        print(f"Task ID: {Task}")
        
        # First attempt to clear the task
        if clear_task_data(base_url, access_token, Task, Task_ID_name):
            continue
            
        # If first attempt failed, try refreshing token
        print(f"Clearing task '{Task_ID_name}' failed. Attempting to refresh token.")
        if refresh_token:
            if refresh_token_function(base_url, refresh_token):
                print("Token refreshed successfully. Retrying clear task.")
                if clear_task_data(base_url, access_token, Task, Task_ID_name):
                    continue
                    
        # If token refresh failed, try logging in again
        print("Token refresh failed. Attempting to log in again.")
        if log_in(base_url, username, password):
            print("Re-login successful. Retrying clear task.")
            if clear_task_data(base_url, access_token, Task, Task_ID_name):
                continue
                
        print(f"Failed to clear task '{Task_ID_name}' after all attempts.")
        print("Please check if:")
        print("1. The task ID is correct")
        print("2. You have the necessary permissions")
        print("3. The task is in a state that can be cleared")
        print("4. The API endpoint is correct (should be /api/v1/tasks/{task_id}/clear)")
        break  # Stop processing if all attempts failed
        
    return None 


In [ ]:
Clear_task (Task_ID)
#Start_task_items(Task_ID)